In [ ]:
# %% [markdown]
# ## Aim:
# How to link the rule-based extracted CI_TYPE-GEO pairs with the respective Ci failure impacts
# 
# **Idea**\
# Test using prompt engineering by passing table of CI-GEO pairs to GPT-J model.
# Steps:
# * Load model and apply it always on one chunk of the document to extract CI failure impacts 
# * Use prompt engineering to extract time and location of the CI failure (origin) the CI impacts (impact location)
# or 
# * Pass dataframe of pairs as input to the model
# or
# * Use few shot prompting with example answers
# 
# **Finally:**
# * Compaire all approaches of spatial and temporal linking CI failure impacts

# %%
import os


# # settings for CUDA and PYTORCH
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ["PYTORCH_ALLOC_CONF"]="expandable_segments:True" ## improve memory allocation

# # settings for debugging CUDA errors (pinpoint exact line of error)
os.environ["TORCH_USE_CUDA_DSA"] = "1"
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1" 

# activate global venv explicitly
os.environ["VIRTUAL_ENV"] = "/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/.venv"


%%
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())  # should give 2
print(torch.cuda.get_device_name())
print(torch.cuda.get_device_properties(0))
# print(torch.cuda.get_device_properties(1))
print(torch.cuda.get_device_capability())
print(torch.cuda.get_arch_list())
print(torch.__version__)
print(torch.version.cuda)


## --> must be CUDA 12.6, torch: 2.91, ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']

import os
import sys
# import subprocess
import re
import time
from glob import glob
from pathlib import Path
import gc
# from io import StringIO
# import json

# from tqdm import tqdm
import numpy as np
import pandas as pd
import geonamescache
# import pyarrow as pa
# import pyarrow.parquet as pq
import spacy
from huggingface_hub import login
from pdfminer.high_level import extract_text
import langdetect
from transformers import AutoTokenizer
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType
# from langchain.document_loaders import DirectoryLoader #, UnstructuredLoader
# from langchain_community.document_loaders import DirectoryLoader, UnstructuredLoader
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    AcceleratorOptions,
    AcceleratorDevice,
)
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline
from docling.document_converter import DocumentConverter, FormatOption
from docling.chunking import HybridChunker



from src.settings import settings as s
import src.document_cleaning as dc
import src.translation_model as tm
import src.extraction_model as em
import src.postprocess as pp
import src.utils as u
from geollama.geollama.main import GeoLlama
from geollama.geollama.model import TopoModel, RAGModel




test_mode = True

# login to HF
login(token=os.getenv("HUGGINGFACE_TOKEN")) 
# NOTE raises exception if not env.variable doesnt exist (compared to os.envrion.get and its shortcut os.getenv)


# NOTE. disabled batch size as OOM for CUDA despite chunkwise memory cleaning, nvtop to find best batchsize
BATCH_SIZE = s.BATCH_SIZE  # max for nvidia GPU

# torch.manual_seed(42)

#  automatic linebreaks and multi-line cells.
pd.set_option('display.max_colwidth', 100000)
pd.set_option("display.colheader_justify", "left")

print(os.environ["CUDA_VISIBLE_DEVICES"])

# clean up before applying CUDA
gc.collect()
torch.cuda.empty_cache() 
print(torch.cuda.memory_reserved() / 1e9)
torch.no_grad()



# ## TODO test to prevent CUDA-OOM when reused
# ## Source: https://spacy.io/usage/embeddings-transformers
# from thinc.api import set_gpu_allocator, require_gpu

# # Use the GPU, with memory allocations directed via PyTorch.
# # This prevents out-of-memory errors that would otherwise occur from competing
# # memory pools.
# set_gpu_allocator("pytorch")
## require_gpu(0)



# %% [markdown]
# ## Set paths and vars

# %%
# set wd to project root
# os.chdir("/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval")

## set path variables
DOCS_DIR = Path(s.PATH_DATA / "text_sources/")
PARSED_TEXT_DIR = Path(s.PATH_DATA / "parsed_documents/")
NER_PATTERNS_FILEPATH = Path(s.NER_PATTERNS_FILEPATH)
LLM_OUTPUTS_DIR = Path(s.PATH_DATA / "llm_outputs/")
geollama_OUTPUTS_DIR = Path(s.PATH_DATA / "geollama_outputs/")

os.makedirs(PARSED_TEXT_DIR, exist_ok=True)
os.makedirs(s.PATH_LLM_DATA, exist_ok=True)
os.makedirs(geollama_OUTPUTS_DIR, exist_ok=True)


# CI GEO pairs
CI_GEO_FILEPATH = Path( s.PATH_DATA / s.CI_GEO_PAIRS_FILENAME)

## store LLM 1 response and prompt
OUTPUT_LLM1_FILEPATH =  Path(s.PATH_LLM_DATA / s.LLM_DATA_FILENAME)
OUTPUT_geollama_FILEPATH =  Path(s.PATH_LLM_DATA / "geollama_results.csv")




# %% [markdown]
# ### Set test mode

# %%

docs_list_sample = [
        # Path(PARSED_TEXT_DIR, "Lloyd's List 2024 - Port of Valencia reopens after devastating floods_cleaned.jsonl"),
        # Path(PARSED_TEXT_DIR, "Containerlift 2024 - Valencia Port Resumes Operations Following Devastating Flooding in Spain - Containerlift.co.uk - Transport_Lifting_Shipping_cleaned.jsonl"), 
        # Path(PARSED_TEXT_DIR, "ABC 2024 - Traffic jams and flight delays due to heavy rain and lightning storm in Malaga_cleaned.jsonl"),

        # Path(PARSED_TEXT_DIR, "Karakatsani 2023 - Greece economy briefing The economic impact of the recent devastating floods in Greece_cleaned.jsonl"),
    # Path(PARSED_TEXT_DIR, "Koks 2022 - Brief communication_cleaned.jsonl"),
        # Path(PARSED_TEXT_DIR, "European Investment Bank 2025 - Spain_ EIB lends €50 million to Iberdrola to rebuild and climate-proof flood-hit power infrastructure in Valencia_cleaned.jsonl"),
        # Path(PARSED_TEXT_DIR, "Wilson 2024 - Flash floods in Spain sweep away cars, disrupt trains and leave several missing _ AP News_cleaned.jsonl"),     
        #     Path(PARSED_TEXT_DIR, "Wildhagen 2013 - Hochwasser_ Wie die Flut Unternehmen lahmlegt_cleaned.jsonl"),
        #     Path(PARSED_TEXT_DIR, "AFP 2022 - The_Vibes_Valencia Airport in Madrid briefly shut as lightning hits runway _ World _ The Vibes_cleaned.jsonl"),
        #     Path(PARSED_TEXT_DIR, "Artemis 2015 - PERILS finalises Storm Desmond UK flood loss estimate at £604m_cleaned.jsonl"),
        #     Path(PARSED_TEXT_DIR, "Brown 2010 - Economy feels chill as UK grinds to a halt _ The Independent_cleaned.jsonl"),
        #     # Path(PARSED_TEXT_DIR, "Diakakis 2020 - A systematic assessment of the effects of extreme flash floods on transportation infrastructure and circulation: The example of the 2017 Mandra flood_cleaned.jsonl"),
        # Path(PARSED_TEXT_DIR, "EFE 2024 - The DANA storm, live_ The death toll rises to 158_cleaned.jsonl"),
        #     # Path(PARSED_TEXT_DIR, "Eurelectric 2006 - Impacts of Severe Storms on Electric Grids_cleaned.jsonl"),
        #     Path(PARSED_TEXT_DIR, "Euronews 2024 - Spain floods_ Death toll rises to 205 as nation braces for more rain _cleaned.jsonl"),
        # Path(PARSED_TEXT_DIR, "Ferlita 2023 - Incendi in Sicilia, ecco cosa accade_cleaned.jsonl"),
        #     Path(PARSED_TEXT_DIR, "Fink 2004 - The 2003 European summer heatwaves and drought - synoptic diagnosis and impacts_cleaned.jsonl"),
        # Path(PARSED_TEXT_DIR, "Gilbody Dickerson 2024 - Spain floods_ At least 95 people killed including British man near Malaga _ World News _ Sky News_cleaned.jsonl"),
        #     # Path(PARSED_TEXT_DIR, "Kadir 2014 - The Impact of Natural Disasters on Critical Infrastructures - A Domino Effect-based Study_cleaned.jsonl"),
        #     Path(PARSED_TEXT_DIR, "Kaur 2025 - Authorities suspect arson in 17 wildfires across Dalmatian coast, Croatia - The Watchers_cleaned.jsonl"),
        #     # Path(PARSED_TEXT_DIR, "Keller 2014 - Mapping Natural Hazard Impacts on Road Infrastructure—The Extreme Precipitation in Baden-Württemberg_cleaned.jsonl"),
        #     Path(PARSED_TEXT_DIR, "Kettle 2020 - Storm Xaver over Europe in December 2013 Overview of energy impacts and North Sea events_cleaned.jsonl"),
        #     # Path(PARSED_TEXT_DIR, "Koks 2019 - Understanding Business Disruption and Economic Losses Due to Electricity Failures and Flooding_cleaned.jsonl"),
        #     # Path(PARSED_TEXT_DIR, "Korzilius 2021 Nach der Flut_cleaned.jsonl"),

    # Path(PARSED_TEXT_DIR, "Khazai 2013 - Juni-Hochwasser 2013 in Mitteleuropa - Fokus Deutschland Bericht 2 Auswirkungen und Bewältigung_cleaned.jsonl"),
    # Path(PARSED_TEXT_DIR, "Rozendaal 2021 - Infrabel_ Flood damage to railway track worth tens of millions of euros _ SpoorPro - incomplete_cleaned.jsonl"),
    # Path(PARSED_TEXT_DIR, "Skoulding 2023 - Where are the fires in Italy today as temperatures rise to 47.6C on Sicily_ _ The Independent_cleaned.jsonl"),
    # Path(PARSED_TEXT_DIR, "The Guardian 2015 - Storm Desmond damage across Cumbria estimated at £500m _ Storm Desmond_cleaned.jsonl"),

    # # long processing
    # Path(PARSED_TEXT_DIR, "Treanor 2015 - Storm Desmond damage across Cumbria estimated at £500m _ Storm Desmond _The Guardian_cleaned.jsonl"),
    Path(PARSED_TEXT_DIR, "AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLUVIAS INTENSAS_cleaned.jsonl"),
    
        #     # not part of valid set:
        #     # Path(PARSED_TEXT_DIR, "Krausmann 2014 - STREST report on lessons learned from recent catastrophic events_cleaned.jsonl"), # > 1800 entries LLMv3.0 incl. hallucinations
]


## Test mode
if test_mode:
    search_path = docs_list_sample
    print("Test mode is ON. Using only a small sample of documents for testing.")
else:
    search_path = glob(str(Path(PARSED_TEXT_DIR, "*cleaned.jsonl")))

print(f"Using {len(search_path)} documents for processing.")



# %% [markdown]
# ###  Load spaCy language model

## RELOAD spacy pipeline
nlp = spacy.load("./spacy_model_pipeline")

# add CI_TYPE patterns to spacy nlp model pipeline
config = {"spans_key": None, "annotate_ents": True, "overwrite": False}
## see for more info: https://spacy.io/usage/rule-based-matching#entityruler
## NOTE EntityRuler is hidden inside .add_pipe()
try:
    ruler = nlp.add_pipe("span_ruler", config=config)
    ruler.from_disk(s.NER_PATTERNS_FILEPATH)
except ValueError:
    print("SpanRuler already exists in pipeline.")
    ruler = nlp.get_pipe("span_ruler")
    ruler.from_disk(s.NER_PATTERNS_FILEPATH)


# load NER patterns for CI types and their subgroups (needed for cleaning LLm response - STEP 1 before continuing with STEP 2)
ci_patterns = pd.read_json("./ner_patterns.jsonl/patterns", lines=True)

# %%


# %% [markdown]
# ### geollama pipeline
# 
# 

# %%
## Make sure that still both GPUS are visible

# print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# print(os.environ["CUDA_VISIBLE_DEVICES"])
# # !nvidia-smi




topo_model = TopoModel(
    model_name='JoeShingleton/GeoLlama-3.2-3b-toponym',
    # model_name='JoeShingleton/GeoLlama_7b_toponym', 
    prompt_path='../geollama/data/prompt_templates/prompt_template.txt',
    instruct_path='../geollama/data/prompt_templates/topo_instruction.txt',
    input_path=None,
    config_path='../geollama/data/config_files/model_config.json'
)

rag_model = RAGModel(
    model_name='JoeShingleton/GeoLlama-3.2-3b-RAG',
    # model_name='JoeShingleton/GeoLlama_7b_RAG',
    prompt_path='../geollama/data/prompt_templates/prompt_template.txt',
    instruct_path='../geollama/data/prompt_templates/rag_instruction.txt',
    input_path='../geollama/data/prompt_templates/rag_input.txt',
    config_path='../geollama/data/config_files/model_config.json')

geo_llama = GeoLlama(
    topo_model = topo_model, 
    rag_model = rag_model, 
    translate_model=None
)



## Document cleaning

In [ ]:
# # Docling OCR pipeline configs

from pathlib import Path
from docling.datamodel.pipeline_options import PdfPipelineOptions, EasyOcrOptions, AcceleratorOptions
from docling.document_converter import DocumentConverter, PdfFormatOption, PipelineOptions, InputFormat


artifacts_path = Path("./docling_artifacts")
artifacts_path.mkdir(exist_ok=True)

# 2b set OCR options
ocr_options = EasyOcrOptions(
    lang=["en", "fr", "de", "es", "it", "nl"],             # or your preferred language(s)
    download_enabled=True    # <-- This enables model downloads!
)
 
# 2c Configure PDF pipeline options (VALID parameters only)
pipeline_opts = PdfPipelineOptions(
    artifacts_path=artifacts_path,
    do_ocr=True,         # Required for text extraction
    do_table_structure=False,  # Disable table analysis if not needed
    ocr_options=ocr_options
)

# 3. Configure PDF format options with region exclusion
pdf_format_option = PdfFormatOption(
    pipeline_options=pipeline_opts,
    reading_order="natural"
)

# 4. Initialize converter
format_options = {
    InputFormat.PDF: pdf_format_option
}


# setup converter for PDF
converter = DocumentConverter(
    format_options=format_options
)

## test OCR converter
# pdf_doc = converter.convert(source=pdf_filepath).document  ## !! recognizes section titles, footers/headers !! :D

# # chunking
# chunk_iter = chunker.chunk(dl_doc=pdf_doc)
# chunks = list(chunk_iter)  # splits at e.g. or 

    
# ## TODO pass here DoclingObj
# texts = get_processed_texts(pdf_doc)  # (normal converter, no OCR) for Koks 2022: headers and foooters are separated as single TextItems  (spearated form acutal text.body)


#### loader - bruise code 

In [ ]:
##########################  HYPHEN CLENAING SOLO  (kkep solo or inc. in DoclingParser class)

## document-wise cleaning 

from typing import List, Dict, Tuple, Optional, Union
import re

# noinspection PyPackageRequirements
import nltk
from haystack.dataclasses import ByteStream
from docling_core.types import DoclingDocument
from docling_core.types.doc import CoordOrigin
from docling_core.types.doc.document import SectionHeaderItem, ListItem, TextItem, DocItem


# Module-level caches (private)
_words_list = None
_lemmatizer = None
_stemmer = None


# ## load nltk libs for handling hyphens
# nltk.download('wordnet')
# nltk.download('omw-1.4')


def get_words_list():
    """Lazily load and cache the NLTK english words list."""
    global _words_list
    if _words_list is None:
        import nltk
        nltk.download('words')
        _words_list = set(nltk.corpus.words.words())
    return _words_list


def get_lemmatizer():
    """Lazily load and cache the WordNetLemmatizer."""
    global _lemmatizer
    if _lemmatizer is None:
        from nltk.stem import WordNetLemmatizer
        _lemmatizer = WordNetLemmatizer()
    return _lemmatizer


def get_stemmer():
    """Lazily load and cache the PorterStemmer."""
    global _stemmer
    if _stemmer is None:
        from nltk.stem import PorterStemmer
        _stemmer = PorterStemmer()
    return _stemmer


def is_valid_word(word):
    """
    Check if a word is valid by comparing it directly and via stemming/lemmatization.
    In detail, it checks if the given word, its stem, or its lemma is inlcuded in the word list downloaded from nltk or the customized list of suffixes.

    Returns True (or the valid modified word) if the word is found,
    otherwise returns False.
    """
    words_list = get_words_list()
    stemmer = get_stemmer()
    lemmatizer = get_lemmatizer()

    stem = stemmer.stem(word)
    if word.lower() in words_list or word in words_list:
        return True
    elif stem in words_list or stem.lower() in words_list:
        return True

    # Check all lemmatizations of the word
    for pos in ['n', 'v', 'a', 'r', 's']:
        lemma = lemmatizer.lemmatize(word, pos=pos)
        if lemma in words_list:
            return True

    # Check for custom lemmatizations
    # noinspection SpellCheckingInspection
    suffixes = {
        "ability": "able",  # testability -> testable
        "ibility": "ible",  # possibility -> possible
        "iness": "y",  # happiness -> happy
        "ity": "e",  # creativity -> create
        "tion": "e",  # creation -> create
        "able": "",  # testable -> test
        "ible": "",  # possible -> poss
        "ing": "",  # running -> run
        "ed": "",  # tested -> test
        "s": ""  # tests -> test
    }
    for suffix, replacement in suffixes.items():
        if word.endswith(suffix):
            stripped_word = word[:-len(suffix)] + replacement
            # Recursively check the modified word; if valid, return the modified form.
            result = is_valid_word(stripped_word)
            if result:
                return result

    return False


def combine_hyphenated_words(p_str):
    """
    Combine hyphenated words if the parts together form a valid word.
    Otherwise, preserve the hyphen (assuming it connects two valid words).
    """

    def replace_dash(match):
        word1, word2 = match.group(1), match.group(2)
        combined = word1.strip() + word2.strip()

        # If there is a space after the hyphen and the combined word is valid,
        # assume the hyphen was splitting a single word.
        if word2.startswith(" ") and is_valid_word(combined):
            return combined
        # If both parts are valid words on their own, keep them hyphenated.
        elif is_valid_word(word1.strip()) and is_valid_word(word2.strip()):
            return word1.strip() + '-' + word2.strip()
        # Otherwise, if the combined word is valid, return it.
        elif is_valid_word(combined):
            return combined
        # If the combined word starts with a capital letter (likely a proper noun)
        # and the second part isn’t valid on its own, combine them.
        elif combined[0].isupper() and not word2.strip()[0].isupper() and not is_valid_word(word2.strip()):
            return combined

        # Default: assume the hyphen is meant to connect two words.
        return word1.strip() + '-' + word2.strip()

    # Replace any soft hyphen characters with a regular dash.
    p_str = p_str.replace("­", "-")
    # Look for hyphens between word parts (with or without an extra space)
    p_str = re.sub(r'(\w+)-(\s?\w+)', replace_dash, p_str)

    return p_str

In [ ]:
### 

def clean_text(p_str: str) -> str:
    p_str = str(p_str).strip()  # Convert text to a string and remove leading/trailing whitespace
    p_str = p_str.encode('utf-8').decode('utf-8')
    p_str = re.sub(r'\s+', ' ', p_str).strip()  # Replace multiple whitespace with single space
    p_str = re.sub(r"([.!?]) '", r"\1'", p_str)  # Remove the space between punctuation (.!?) and '
    p_str = re.sub(r'([.!?]) "', r'\1"', p_str)  # Remove the space between punctuation (.!?) and "
    p_str = re.sub(r'\s+\)', ')', p_str)  # Remove whitespace before a closing parenthesis
    p_str = re.sub(r'\s+]', ']', p_str)  # Remove whitespace before a closing square bracket
    p_str = re.sub(r'\s+}', '}', p_str)  # Remove whitespace before a closing curly brace
    p_str = re.sub(r'\s+,', ',', p_str)  # Remove whitespace before a comma
    p_str = re.sub(r'\(\s+', '(', p_str)  # Remove whitespace after an opening parenthesis
    p_str = re.sub(r'\[\s+', '[', p_str)  # Remove whitespace after an opening square bracket
    p_str = re.sub(r'\{\s+', '{', p_str)  # Remove whitespace after an opening curly brace
    p_str = re.sub(r'(?<=\s)\.([a-zA-Z])', r'\1',
                   p_str)  # Remove a period that follows a whitespace and comes before a letter
    p_str = re.sub(r'\s+\.', '.', p_str)  # Remove any whitespace before a period

    # Remove footnote numbers at end of a sentence. Check for a digit at the end and drop it
    # until there are no more digits or the sentence is now a valid end of a sentence.
    while p_str and p_str[-1].isdigit() and not is_sentence_end(p_str):
        p_str = p_str[:-1].strip()
    
    return p_str

##########################
def remove_figure_references(p_str: str) -> str:
    # remove potneital figure reference when they are colsed by bracketss, e.g. (A1), (B20)
    # this is done to avoid mismatches with road names
    p_str = re.sub(r"\s+\([A-Z][0-9]{1,}\)\s", "", p_str)
    return p_str

def is_reference_section(document_text: str) -> bool:
    # search for reference section
    pattern = re.compile(
        r"^(References|REFERENCES|Bibliography|BIBLIOGRAPHY)$", flags=re.MULTILINE
    )
    # re.MULTILINE in combination with "^" and case sensitive : find search words only when they are at beginning of a new line
    matches = re.findall(pattern, document_text)
    if matches:
        print(f"Reference section found!" )
        return True
###################



def is_sentence_end(text: str) -> bool:
    has_end_punctuation: bool = is_ends_with_punctuation(text)
    # Does it end with a closing bracket, quote, etc.?
    ends_with_bracket: bool = (text.endswith(")")
                               or text.endswith("]")
                               or text.endswith("}")
                               or text.endswith("\"")
                               or text.endswith("\'"))
    return (has_end_punctuation or
            (ends_with_bracket and is_ends_with_punctuation(text[0:-1])))


def combine_paragraphs(p1_str: str, p2_str: str):
    # If the paragraph ends without final punctuation, combine it with the next paragraph
    if is_sentence_end(p1_str):
        return p1_str + "\n" + p2_str
    else:
        return p1_str + " " + p2_str



def is_section_header(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
    if text is None:
        return False
    return text.label == "section_header"


def is_page_footer(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
    return text.label == "page_footer"


def is_page_header(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
    return text.label == "page_header"


def is_footnote(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
    return text.label == "footnote"


def is_list_item(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
    return text.label == "list_item"


def is_text_break(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
    return is_page_header(text) or is_section_header(text) or is_footnote(text)


def is_page_not_text(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
    return text.label not in ["text", "list_item", "formula"]


def is_page_text(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
    return not is_page_not_text(text)


def is_ends_with_punctuation(text: str) -> bool:
    return text.endswith(".") or text.endswith("?") or text.endswith("!")


def is_too_short(doc_item: DocItem, threshold: int = 2) -> bool:
    return doc_item.label == "text" and len(doc_item.text) <= threshold


def is_bottom_note(text: DocItem, near_bottom: bool = False) -> bool:
    # if 'Morgenstern was then the director' in text.text:
    #     pass
    # if text.text.startswith("10. Summing up o f"):
    #     pass

    # If it is specifically digits followed by a period, followed by a space, and it is
    # a section header or a list item, then it is NOT a bottom note
    if bool(re.match(r"^\d+\.\s", text.text)) and (is_section_header(text) or is_list_item(text)):
        return False
    # If it's digits followed by a letter without a space then it's a bottom note
    if bool(re.match(r"^\d+[A-Za-z]", text.text)):
        return True

    if text is None or not is_page_text(text):
        return False
    # Check for · at the beginning of the line. This is often how OCR represents footnote number.
    if text.text.startswith("·") and not text.text.startswith("· "):
        return True

    if re.match(r"^\d", text.text):
        # If the first digit is zero, it can't be a footnote because that should never happen.
        if text.text.startswith("0"):
            return False
        if near_bottom:
            # Check if this is three digits with the third digit being a 1 followed by a space
            # This is usually where the last 1 was supposed to be an 'I'.
            return re.match(r"^\d{1,2}1 ", text.text) or not is_list_item(text)

    return False


def is_near_bottom(doc_item: DocItem, same_page_items: [DocItem], threshold: float = 0.3) -> bool:
    """
    Determine if a DocItem is near the bottom of its page.

    Parameters:
    - doc_item: The DocItem object containing provenance data with 'bbox'.
    - doc: The DoclingDocument containing all DocItems.
    - threshold: Distance in points from the bottom to consider as 'near the bottom'.

    Returns:
    - True if the DocItem is within the threshold from the bottom, False otherwise.
    """
    # Check if the DocItem has provenance data with a bounding box
    if hasattr(doc_item.prov[0], 'bbox'):
        bbox = doc_item.prov[0].bbox
    else:
        return False  # No bounding box available

    # Extract the coordinate origin and bounding box coordinates
    coord_origin = bbox.coord_origin
    x0, y0, x1, y1 = bbox.l, bbox.b, bbox.r, bbox.t

    # Find the maximum y1 value on the page
    page_top: float = max(item.prov[0].bbox.t for item in same_page_items if hasattr(item.prov[0], 'bbox'))
    # Find the min y1 value on the page
    page_bottom: float = min(item.prov[0].bbox.b for item in same_page_items if hasattr(item.prov[0], 'bbox'))
    page_size: float = page_top - page_bottom
    # Threshold is page_bottom + (size of page * threshold amount) (i.e. % of page to be considered the 'bottom')
    bottom_threshold: float = page_bottom + (page_size * threshold)

    if coord_origin == CoordOrigin.BOTTOMLEFT:
        # In this system, y1 is the distance from the top of the paragraph to the bottom of the page
        return y1 <= bottom_threshold
    elif coord_origin == CoordOrigin.TOPLEFT:
        # In this system, y1 is the distance from the top of the paragraph to the top of the page
        return y1 >= bottom_threshold
    else:
        raise ValueError("Unknown coordinate origin.")



def is_text_item(item: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
    return not (is_section_header(item)
                or is_page_footer(item)
                or is_page_header(item)
                # or is_reference_section(item)
            )



def get_next_text(texts: List[Union[SectionHeaderItem, ListItem, TextItem]], i: int) \
        -> Optional[Union[ListItem, TextItem]]:
    # Seek through the list of texts to find the next text item using is_text_item
    # Should return None if no more text items are found
    for j in range(i + 1, len(texts)):
        if j < len(texts) and is_text_item(texts[j]):  # skips page headers/footers
            return texts[j]
    return None


def is_roman_numeral(s: str) -> bool:
    roman_numeral_pattern = r'(?i)^(M{0,3})(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})$'
    return bool(re.match(roman_numeral_pattern, s.strip()))

def should_skip_element(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
    return any([
        is_page_footer(text),
        is_page_header(text),
        is_roman_numeral(text.text)
    ])


def get_processed_texts(doc: DoclingDocument) -> List[DocItem]:
    """
    Processes the document's text items page by page, separating regular content from notes
    (footnotes and bottom notes), and returns a list of DocItems with notes at the end.
    """
    regular_texts: List[DocItem] = []
    notes: List[DocItem] = []
    processed_pages: set[int] = set()  # Keep track of processed pages
    reached_bottom_notes: bool = False
    same_page_items: List[DocItem] = []
    near_bottom: bool = False
    mislabeled: List[DocItem] = []

    for text_item in doc.texts:
        page_number = text_item.prov[0].page_no

        if page_number not in processed_pages:
            # On new page, so get all items on the current page
            same_page_items = [
                item for item in doc.texts if item.prov[0].page_no == page_number
            ]
            processed_pages.add(page_number)  # Mark the page as processed
            reached_bottom_notes = False

        if not reached_bottom_notes:
            near_bottom = is_near_bottom(text_item, same_page_items, threshold=0.5)

        if is_too_short(text_item):
            continue
        elif reached_bottom_notes or is_footnote(text_item):
            notes.append(text_item)
        elif is_bottom_note(text_item, near_bottom=near_bottom):
            notes.append(text_item)
            reached_bottom_notes = True
        else:
            regular_texts.append(text_item)

        # Check if the DocItem is a SectionHeaderItem. If so, turn it into a TextItem.
        if reached_bottom_notes and is_section_header(text_item):
            mislabeled.append(text_item)

    return regular_texts + notes


def add_paragraph(
    text: str,               
    # para_num: int, section: str, page: Optional[int], 
    docs: List[ByteStream], 
    # meta: List[Dict]
):
    docs.append(ByteStream(text.encode('utf-8')))
    # meta.append({
    #     **meta_data,
    #     # "paragraph_#": str(para_num),
    #     "section_name": section,
    #     "page_#": str(page)
    # })




In [ ]:
### test solo func from Bruise

pdf_filename = search_path[0]
pdf_filepath = os.path.join(DOCS_DIR, pdf_filename.name.replace("_cleaned.jsonl", ".pdf") )

# load tokenizer
# embed_model =  "sentence-transformers/all-MiniLM-L6-v2"
# tokenizer = HuggingFaceTokenizer(
#     tokenizer=AutoTokenizer.from_pretrained(embed_model),
#     max_tokens=256, # max tokens for MiniLM-l6-v2, set here explicitly
#     # standardize input sizes of chunks for Llama models
#     padding=True, # add zero as extra tokens to too short sequences so that they have the same length as other chunks
#     truncation=True, # truncates too long sequences (> max_tokens). If False, they will be split into multiple chunks
# )

# ## init chunker - based on hierachical chunker but also considers max token leng, merge smaller chunks, except when at end of paragraph (merge_peers=True)
# chunker = HybridChunker(
#     tokenizer=tokenizer,
#         # max_tokens=256, # max tokens for MiniLM-l6-v2, set here explicitly
#         # chunk_overlap=0, # no overlap between chunks, as we use merge_peers to merge smaller chunks and avoid splits in sentences
#     split_by_sentence=True, # split by sentence first before merging smaller chunks, to avoid splits in sentence middle
#     merge_peers=True,  # optional, defaults to True
# )

## test OCR converter
print("Using OCR for text extraction as it identifies section titles, footers/headers and pagenumbers as such, but reads in also figure text sometimes") 
# NOTE all other standard doclingConverter retunr section/headers etc as BODY not FURNITURE
# NOTE: partly reads in figure text
pdf_doc = converter.convert(source=pdf_filepath).document  ## !! recognizes section titles, footers/headers !! :D


mislabeled: List[DocItem] = []
min_paragraph_size = 100  
temp_docs: List[ByteStream] = []
temp_meta: List[Dict[str, str]] = []
i: int
combined_paragraph: str = ""
combined_chars: int = 0
para_num: int = 0
section_name: str = ""
page_no: Optional[int] = None
first_note: bool = False


src_language_doc = langdetect.detect(pdf_filename.stem.replace(".pdf", "").lower())  # lower case improves language detection

## Load only Doc.items, not its further meta data
texts = get_processed_texts(pdf_doc)  


for i, text in enumerate(texts):

    # get next text only when it is not page header/footer
    next_text = get_next_text(texts, i)
    # page_no = get_current_page(text, combined_paragraph, page_no)


    # Update section header if the element is a section header
    # TODO: Need a stronger check on section headers that takes top of page into account, etc
    if is_section_header(text) and text not in mislabeled:
        print("!!  Section header found:", text.text)
        section_name = text.text
        continue

    if is_reference_section(section_name):
        print("Reference section found. Stopping further processing of document.")
        break  

    if should_skip_element(text):
        continue
    
    # clean from double wihitespace, newlines, etc.
    p_str = clean_text(text.text)

    # clean from potential figure references
    p_str = remove_figure_references(p_str)

    # Removing URLs 
    # LangExtract tries to open these URLs when they occur in the document text
    # p_str= re.sub(r"http\S+", "", p_str) 

    p_str_chars = len(p_str)

    # If the paragraph does not end with final punctuation, accumulate it
    if not is_sentence_end(p_str):
        combined_paragraph = combine_paragraphs(combined_paragraph, p_str)
        combined_chars += p_str_chars
        continue

    # p_str ends with a sentence end; decide whether to process or accumulate it
    total_chars = combined_chars + p_str_chars
    if is_section_header(next_text):
        # Immediately process if the next text is a section header
        p_str = combine_paragraphs(combined_paragraph, p_str)
        combined_paragraph, combined_chars = "", 0
    elif total_chars < min_paragraph_size:
        # Not enough characters accumulated yet; decide based on next_text
        if next_text is None or (not is_page_text(next_text) and is_sentence_end(p_str)):
            # End of document or next text item is not a text item and current paragraph ends with punctuation
            # Process the paragraph and reset the accumulator even though this is a short paragraph
            p_str = combine_paragraphs(combined_paragraph, p_str)
            combined_paragraph, combined_chars = "", 0
        else:
            # Combine with next paragraph
            combined_paragraph = combine_paragraphs(combined_paragraph, p_str)
            combined_chars = total_chars
            continue
    else:
        # Sufficient characters: process the paragraph and reset the accumulator
        p_str = combine_paragraphs(combined_paragraph, p_str)
        combined_paragraph, combined_chars = "", 0

    p_str = combine_hyphenated_words(p_str)
    if p_str:  # Only add non-empty content
        para_num += 1
        add_paragraph(
            p_str, 
            #para_num, section_name, page_no, 
            temp_docs, 
            # temp_meta
        )
        page_no = None
    
    print("\n-__ Paragraph #", para_num, "; Section:", section_name)
    print(p_str)
    # print(temp_docs, temp_meta)

    ## Translation
    if src_language_doc != "en":

        supported_languages = ["fr", "de", "es", "it", "itc", "nl"]
        if src_language_doc not in supported_languages:
            print(f"Unsupported source language: {src_language_doc}. Continue with extraction on original text")
            continue 

        print(f"Translating {src_language_doc} --> en")
        text_transl = tm.translate_2_english(src_language_doc, p_str)     
        
        p_str = text_transl   

    ## STEP 2: then do chunking with HybridChunker  TODO
    # # NOTE maybe only a bit needed as already very good splits but maybe tokensizes need to be adapted 
    ## TODO inlcude p_str always as doclingobj part or write it back to DoclingObject, (maybe with section_name as meta info)

    # # chunking
    # chunk_iter = chunker.chunk(dl_doc=pdf_doc)
    # chunks = list(chunk_iter)




##### DOCUMENTATION - OCR 
* reads in table and figure texts 
* e.g AEMET sometimes mixes paragraphs - not removes them but places them in other TextItems and with wrong section.titles)

In [ ]:
matches?

In [ ]:
# for pdf_filename in os.listdir(DOCS_DIR):
for pdf_filename in search_path:
    # if pdf_filename.endswith(".pdf"):

        md_filename = f"{Path(pdf_filename).stem}.md"

        #NOTE dummy for dev and optimizing chunking and text7doc cleaning
        # pdf_filepath = os.path.join(DOCS_DIR, Path(pdf_filename))
        pdf_filepath = os.path.join(DOCS_DIR, pdf_filename.name.replace("_cleaned.jsonl", ".pdf") )


        pdf_text = extract_text(pdf_filepath)

        ## Translation
        src_language_doc = langdetect.detect(pdf_filename.stem.replace(".pdf", "").lower())  # lower case improves language detection

        if src_language_doc != "en":
            
            # untidy text cleaning (only for translation)
            text_cleaned = re.sub(r"([^\s-])\n([^\s-])", r"\1 \2", pdf_text) # replace linebreak symbols when they occur just once, with whitespace (two linebreaks - probably new subsection)
            text_cleaned = text_cleaned.replace("/\n{2,}/g", "\n")  # replace multiple linebreaks (e.g. before subsection), by just one linebreak
            # Matches \n not preceded or followed by \n
            text_cleaned = re.sub(r"\s+", " ", text_cleaned)  # replace >1 whitespaces with single whitespace
            text_cleaned = re.sub(r"([^\s-])- ([^\s-])", r"\1-\2", text_cleaned)  # remove hyphens+whitespace in the middle of lines (keep only hyphen)
            
            supported_languages = ["fr", "de", "es", "it", "itc", "nl"]
            if src_language_doc not in supported_languages:
                print(f"Unsupported source language: {src_language_doc}. Continue with extraction on original text")
                continue 

            print(f"Translating {src_language_doc} --> en")
            text_transl = tm.translate_2_english(src_language_doc, text_cleaned)     
            pdf_text = text_transl   
            

        ## tidy text cleaning (all in english)


        # <<<-- HERE include DOCLINGPARSER TODO
        

        print("Remove hyphens when it is a linebreak etc.")
        # check if two sides of a hyphen contain words or if the hyphen is a linebreak etc.
        pdf_text = combine_hyphenated_words(pdf_text)

        ## remove potential figure references (e.g. (A1), (B20)) to avoid mismatches with road names
        print("Remove potential figure references")
        pdf_text = re.sub(r"\s+\([A-Z][0-9]{1,}\)\s", " ", pdf_text) 

        print("Remove reference section")
        pdf_text = dc.remove_references(pdf_text)

        print("Removing URLs") # LangExtract tries to open these URLs when they occur in the document text
        pdf_text = re.sub(r"http\S+", "", pdf_text) 

        
pdf_text[1000:5000]

In [ ]:
# !uv add haystack-ai
# !uv lock
# !uv sync


#### loader bruise copied

In [ ]:
# ## copied from https://github.com/brucenielson/BookSearchArchive/blob/e2d6c4145d7931648d5854ba29186cbec8150e87/docling_parser.py

# from typing import List, Dict, Tuple, Optional, Union
# import re
# # noinspection PyPackageRequirements
# from haystack.dataclasses import ByteStream
# from docling_core.types import DoclingDocument
# from docling_core.types.doc import CoordOrigin
# from docling_core.types.doc.document import SectionHeaderItem, ListItem, TextItem, DocItem

# # Module-level caches (private)
# _words_list = None
# _lemmatizer = None
# _stemmer = None


# def get_words_list():
#     """Lazily load and cache the NLTK words list."""
#     global _words_list
#     if _words_list is None:
#         import nltk
#         nltk.download('words')
#         _words_list = set(nltk.corpus.words.words())
#     return _words_list


# def get_lemmatizer():
#     """Lazily load and cache the WordNetLemmatizer."""
#     global _lemmatizer
#     if _lemmatizer is None:
#         from nltk.stem import WordNetLemmatizer
#         _lemmatizer = WordNetLemmatizer()
#     return _lemmatizer


# def get_stemmer():
#     """Lazily load and cache the PorterStemmer."""
#     global _stemmer
#     if _stemmer is None:
#         from nltk.stem import PorterStemmer
#         _stemmer = PorterStemmer()
#     return _stemmer


# def is_valid_word(word):
#     """
#     Check if a word is valid by comparing it directly and via stemming/lemmatization.

#     Returns True (or the valid modified word) if the word is found,
#     otherwise returns False.
#     """
#     words_list = get_words_list()
#     stemmer = get_stemmer()
#     lemmatizer = get_lemmatizer()

#     stem = stemmer.stem(word)
#     if word.lower() in words_list or word in words_list:
#         return True
#     elif stem in words_list or stem.lower() in words_list:
#         return True

#     # Check all lemmatizations of the word
#     for pos in ['n', 'v', 'a', 'r', 's']:
#         lemma = lemmatizer.lemmatize(word, pos=pos)
#         if lemma in words_list:
#             return True

#     # Check for custom lemmatizations
#     # noinspection SpellCheckingInspection
#     suffixes = {
#         "ability": "able",  # testability -> testable
#         "ibility": "ible",  # possibility -> possible
#         "iness": "y",  # happiness -> happy
#         "ity": "e",  # creativity -> create
#         "tion": "e",  # creation -> create
#         "able": "",  # testable -> test
#         "ible": "",  # possible -> poss
#         "ing": "",  # running -> run
#         "ed": "",  # tested -> test
#         "s": ""  # tests -> test
#     }
#     for suffix, replacement in suffixes.items():
#         if word.endswith(suffix):
#             stripped_word = word[:-len(suffix)] + replacement
#             # Recursively check the modified word; if valid, return the modified form.
#             result = is_valid_word(stripped_word)
#             if result:
#                 return result

#     return False


# def combine_hyphenated_words(p_str):
#     """
#     Combine hyphenated words if the parts together form a valid word.
#     Otherwise, preserve the hyphen (assuming it connects two valid words).
#     """

#     def replace_dash(match):
#         word1, word2 = match.group(1), match.group(2)
#         combined = word1.strip() + word2.strip()

#         # If there is a space after the hyphen and the combined word is valid,
#         # assume the hyphen was splitting a single word.
#         if word2.startswith(" ") and is_valid_word(combined):
#             return combined
#         # If both parts are valid words on their own, keep them hyphenated.
#         elif is_valid_word(word1.strip()) and is_valid_word(word2.strip()):
#             return word1.strip() + '-' + word2.strip()
#         # Otherwise, if the combined word is valid, return it.
#         elif is_valid_word(combined):
#             return combined
#         # If the combined word starts with a capital letter (likely a proper noun)
#         # and the second part isn’t valid on its own, combine them.
#         elif combined[0].isupper() and not word2.strip()[0].isupper() and not is_valid_word(word2.strip()):
#             return combined

#         # Default: assume the hyphen is meant to connect two words.
#         return word1.strip() + '-' + word2.strip()

#     # Replace any soft hyphen characters with a regular dash.
#     p_str = p_str.replace("­", "-")
#     # Look for hyphens between word parts (with or without an extra space)
#     p_str = re.sub(r'(\w+)-(\s?\w+)', replace_dash, p_str)

#     return p_str


# def is_section_header(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
#     if text is None:
#         return False
#     return text.label == "section_header"


# def is_page_footer(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
#     return text.label == "page_footer"


# def is_page_header(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
#     return text.label == "page_header"


# def is_footnote(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
#     return text.label == "footnote"


# def is_list_item(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
#     return text.label == "list_item"


# def is_text_break(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
#     return is_page_header(text) or is_section_header(text) or is_footnote(text)


# def is_page_not_text(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
#     return text.label not in ["text", "list_item", "formula"]


# def is_page_text(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
#     return not is_page_not_text(text)


# def is_ends_with_punctuation(text: str) -> bool:
#     return text.endswith(".") or text.endswith("?") or text.endswith("!")


# def is_near_bottom(doc_item: DocItem, same_page_items: [DocItem], threshold: float = 0.3) -> bool:
#     """
#     Determine if a DocItem is near the bottom of its page.

#     Parameters:
#     - doc_item: The DocItem object containing provenance data with 'bbox'.
#     - doc: The DoclingDocument containing all DocItems.
#     - threshold: Distance in points from the bottom to consider as 'near the bottom'.

#     Returns:
#     - True if the DocItem is within the threshold from the bottom, False otherwise.
#     """
#     # Check if the DocItem has provenance data with a bounding box
#     if hasattr(doc_item.prov[0], 'bbox'):
#         bbox = doc_item.prov[0].bbox
#     else:
#         return False  # No bounding box available

#     # Extract the coordinate origin and bounding box coordinates
#     coord_origin = bbox.coord_origin
#     x0, y0, x1, y1 = bbox.l, bbox.b, bbox.r, bbox.t

#     # Find the maximum y1 value on the page
#     page_top: float = max(item.prov[0].bbox.t for item in same_page_items if hasattr(item.prov[0], 'bbox'))
#     # Find the min y1 value on the page
#     page_bottom: float = min(item.prov[0].bbox.b for item in same_page_items if hasattr(item.prov[0], 'bbox'))
#     page_size: float = page_top - page_bottom
#     # Threshold is page_bottom + (size of page * threshold amount) (i.e. % of page to be considered the 'bottom')
#     bottom_threshold: float = page_bottom + (page_size * threshold)

#     if coord_origin == CoordOrigin.BOTTOMLEFT:
#         # In this system, y1 is the distance from the top of the paragraph to the bottom of the page
#         return y1 <= bottom_threshold
#     elif coord_origin == CoordOrigin.TOPLEFT:
#         # In this system, y1 is the distance from the top of the paragraph to the top of the page
#         return y1 >= bottom_threshold
#     else:
#         raise ValueError("Unknown coordinate origin.")


# def is_smaller_text(doc_item: DocItem, doc: DoclingDocument, threshold: float = 0.8) -> bool:
#     """
#     Determine if a DocItem's text is smaller than the average text size on its page.

#     Parameters:
#     - doc_item: The DocItem object containing provenance data with 'bbox'.
#     - doc: The DoclingDocument containing all DocItems.
#     - threshold: Ratio of the average text size to consider as 'smaller text'.

#     Returns:
#     - True if the DocItem's text is smaller than the average text size, False otherwise.
#     """
#     # Check if the DocItem has provenance data with a bounding box
#     if hasattr(doc_item.prov[0], 'bbox'):
#         bbox = doc_item.prov[0].bbox
#     else:
#         return False  # No bounding box available

#     # Extract the bounding box coordinates
#     x0, y0, x1, y1 = bbox.l, bbox.b, bbox.r, bbox.t

#     # Calculate the area of the DocItem's bounding box
#     doc_item_area = (x1 - x0) * (y1 - y0)

#     # Filter doc_items that are on the same page
#     same_page_items = [item for item in doc.texts if item.prov[0].page_no == doc_item.prov[0].page_no]

#     # Calculate the average area of bounding boxes on the page
#     total_area = sum(
#         (item.prov[0].bbox.r - item.prov[0].bbox.l) * (item.prov[0].bbox.t - item.prov[0].bbox.b)
#         for item in same_page_items if hasattr(item.prov[0], 'bbox')
#     )
#     num_items = sum(1 for item in same_page_items if hasattr(item.prov[0], 'bbox'))
#     average_area = total_area / num_items if num_items > 0 else 0

#     # Compare the DocItem's area to the average
#     return doc_item_area < average_area * threshold


# def is_too_short(doc_item: DocItem, threshold: int = 2) -> bool:
#     return doc_item.label == "text" and len(doc_item.text) <= threshold


# def is_bottom_note(text: DocItem,
#                    near_bottom: bool = False) -> bool:
#     if 'Morgenstern was then the director' in text.text:
#         pass
#     if text.text.startswith("10. Summing up o f"):
#         pass

#     # If it is specifically digits followed by a period, followed by a space, and it is
#     # a section header or a list item, then it is NOT a bottom note
#     if bool(re.match(r"^\d+\.\s", text.text)) and (is_section_header(text) or is_list_item(text)):
#         return False
#     # If it's digits followed by a letter without a space then it's a bottom note
#     if bool(re.match(r"^\d+[A-Za-z]", text.text)):
#         return True

#     if text is None or not is_page_text(text):
#         return False
#     # Check for · at the beginning of the line. This is often how OCR represents footnote number.
#     if text.text.startswith("·") and not text.text.startswith("· "):
#         return True

#     if re.match(r"^\d", text.text):
#         # If the first digit is zero, it can't be a footnote because that should never happen.
#         if text.text.startswith("0"):
#             return False
#         if near_bottom:
#             # Check if this is three digits with the third digit being a 1 followed by a space
#             # This is usually where the last 1 was supposed to be an 'I'.
#             return re.match(r"^\d{1,2}1 ", text.text) or not is_list_item(text)

#     return False


# def is_sentence_end(text: str) -> bool:
#     has_end_punctuation: bool = is_ends_with_punctuation(text)
#     # Does it end with a closing bracket, quote, etc.?
#     ends_with_bracket: bool = (text.endswith(")")
#                                or text.endswith("]")
#                                or text.endswith("}")
#                                or text.endswith("\"")
#                                or text.endswith("\'"))
#     return (has_end_punctuation or
#             (ends_with_bracket and is_ends_with_punctuation(text[0:-1])))


# def is_text_item(item: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
#     return not (is_section_header(item)
#                 or is_page_footer(item)
#                 or is_page_header(item))


# def get_next_text(texts: List[Union[SectionHeaderItem, ListItem, TextItem]], i: int) \
#         -> Optional[Union[ListItem, TextItem]]:
#     # Seek through the list of texts to find the next text item using is_text_item
#     # Should return None if no more text items are found
#     for j in range(i + 1, len(texts)):
#         if j < len(texts) and is_text_item(texts[j]):
#             return texts[j]
#     return None


# def remove_extra_whitespace(text: str) -> str:
#     # Remove extra whitespace in the middle of the text
#     return ' '.join(text.split())


# def combine_paragraphs(p1_str: str, p2_str: str):
#     # If the paragraph ends without final punctuation, combine it with the next paragraph
#     if is_sentence_end(p1_str):
#         return p1_str + "\n" + p2_str
#     else:
#         return p1_str + " " + p2_str


# def is_roman_numeral(s: str) -> bool:
#     roman_numeral_pattern = r'(?i)^(M{0,3})(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})$'
#     return bool(re.match(roman_numeral_pattern, s.strip()))


# def get_current_page(text: Union[SectionHeaderItem, ListItem, TextItem],
#                      combined_paragraph: str,
#                      current_page: Optional[int]) -> Optional[int]:
#     return text.prov[0].page_no if current_page is None or combined_paragraph == "" else current_page


# def should_skip_element(text: Union[SectionHeaderItem, ListItem, TextItem]) -> bool:
#     return any([
#         is_page_footer(text),
#         is_page_header(text),
#         is_roman_numeral(text.text)
#     ])

# ################################################


# class DoclingParser:
#     def __init__(self, doc: DoclingDocument,
#                  meta_data: dict[str, str],
#                  min_paragraph_size: int = 100, # 300,
#                  start_page: Optional[int] = None,
#                  end_page: Optional[int] = None,
#                  double_notes: bool = False):
#         self._doc: DoclingDocument = doc
#         self._min_paragraph_size: int = min_paragraph_size
#         self._docs_list: List[ByteStream] = []
#         self._meta_list: List[Dict[str, str]] = []
#         self._meta_data: dict[str, str] = meta_data
#         self._start_page: Optional[int] = start_page
#         self._end_page: Optional[int] = end_page
#         self._double_notes: bool = double_notes
#         self._mislabeled: List[DocItem] = []

#     def run(self) -> Tuple[List[ByteStream], List[Dict[str, str]]]:
#         temp_docs: List[ByteStream] = []
#         temp_meta: List[Dict[str, str]] = []
#         combined_paragraph: str = ""
#         i: int
#         combined_chars: int = 0
#         para_num: int = 0
#         section_name: str = ""
#         page_no: Optional[int] = None
#         first_note: bool = False

#         texts = self._get_processed_texts()

#         for i, text in enumerate(texts):

#             ## these sentence snippets dont show up in the used docs
#             # if 'The preceding section was' in text.text:
#             #     # Thinks this is a section header
#             #     pass
#             # if 'Indeed, (cc) is an immediate consequence' in text.text:
#             #     # Doesn't break this section up right
#             #     pass
#             # if '(iii) More generally even' in text.text:
#             #     # Inappropriately sent to footnotes
#             #     pass
#             # if 'What seduces so many people' in text.text:
#             #     # This whole section seems messed up in paragraphs before and after.
#             #     pass
#             # if '18. A' in text.text:
#             #     pass
#             # if text.text.startswith('In order to show that these'):
#             #     # Stops before the one above
#             #     pass

#             next_text = get_next_text(texts, i)
#             page_no = get_current_page(text, combined_paragraph, page_no)

#             # Check if the current page is within the valid range
#             if self._start_page is not None and page_no is not None and page_no < self._start_page:
#                 page_no = None
#                 continue
#             if self._end_page is not None and page_no is not None and page_no > self._end_page:
#                 if self._double_notes and not first_note:
#                     self._min_paragraph_size *= 2
#                     first_note = True
#                 continue

#             # Update section header if the element is a section header
#             # TODO: Need a stronger check on section headers that takes top of page into account, etc
#             if is_section_header(text) and text not in self._mislabeled:
#                 section_name = text.text
#                 continue

#             if should_skip_element(text):
#                 continue
            
#             # clean from double wihitespace, newlines, etc.
#             p_str = clean_text(text.text)

#             # clean from potential figure references
#             p_str = remove_figure_references(p_str)

#             p_str_chars = len(p_str)

#             # If the paragraph does not end with final punctuation, accumulate it
#             if not is_sentence_end(p_str):
#                 combined_paragraph = combine_paragraphs(combined_paragraph, p_str)
#                 combined_chars += p_str_chars
#                 continue

#             # p_str ends with a sentence end; decide whether to process or accumulate it
#             total_chars = combined_chars + p_str_chars
#             if is_section_header(next_text):
#                 # Immediately process if the next text is a section header
#                 p_str = combine_paragraphs(combined_paragraph, p_str)
#                 combined_paragraph, combined_chars = "", 0
#             elif total_chars < self._min_paragraph_size:
#                 # Not enough characters accumulated yet; decide based on next_text
#                 if next_text is None or (not is_page_text(next_text) and is_sentence_end(p_str)):
#                     # End of document or next text item is not a text item and current paragraph ends with punctuation
#                     # Process the paragraph and reset the accumulator even though this is a short paragraph
#                     p_str = combine_paragraphs(combined_paragraph, p_str)
#                     combined_paragraph, combined_chars = "", 0
#                 else:
#                     # Combine with next paragraph
#                     combined_paragraph = combine_paragraphs(combined_paragraph, p_str)
#                     combined_chars = total_chars
#                     continue
#             else:
#                 # Sufficient characters: process the paragraph and reset the accumulator
#                 p_str = combine_paragraphs(combined_paragraph, p_str)
#                 combined_paragraph, combined_chars = "", 0

#             p_str = combine_hyphenated_words(p_str)
#             if p_str:  # Only add non-empty content
#                 para_num += 1
#                 self._add_paragraph(p_str, para_num, section_name, page_no, temp_docs, temp_meta)
#                 page_no = None

#         return temp_docs, temp_meta

#     def _get_processed_texts(self) -> List[DocItem]:
#         """
#         Processes the document's text items page by page, separating regular content from notes
#         (footnotes and bottom notes), and returns a list of DocItems with notes at the end.
#         """
#         regular_texts: List[DocItem] = []
#         notes: List[DocItem] = []
#         processed_pages: set[int] = set()  # Keep track of processed pages
#         reached_bottom_notes: bool = False
#         same_page_items: List[DocItem] = []
#         near_bottom: bool = False

#         for text_item in self._doc.texts:
#             page_number = text_item.prov[0].page_no

#             if page_number not in processed_pages:
#                 # On new page, so get all items on the current page
#                 same_page_items = [
#                     item for item in self._doc.texts if item.prov[0].page_no == page_number
#                 ]
#                 processed_pages.add(page_number)  # Mark the page as processed
#                 reached_bottom_notes = False

#             if not reached_bottom_notes:
#                 near_bottom = is_near_bottom(text_item, same_page_items, threshold=0.5)

#             if is_too_short(text_item):
#                 continue
#             elif reached_bottom_notes or is_footnote(text_item):
#                 notes.append(text_item)
#             elif is_bottom_note(text_item, near_bottom=near_bottom):
#                 notes.append(text_item)
#                 reached_bottom_notes = True
#             else:
#                 regular_texts.append(text_item)

#             # Check if the DocItem is a SectionHeaderItem. If so, turn it into a TextItem.
#             if reached_bottom_notes and is_section_header(text_item):
#                 self._mislabeled.append(text_item)

#         return regular_texts + notes

#     def _add_paragraph(self, text: str, para_num: int, section: str,
#                        page: Optional[int], docs: List[ByteStream], meta: List[Dict]):
#         docs.append(ByteStream(text.encode('utf-8')))
#         meta.append({
#             **self._meta_data,
#             # "paragraph_#": str(para_num),
#             "section_name": section,
#             "page_#": str(page)
#         })


In [ ]:
# pdf_filepath = os.path.join(DOCS_DIR, pdf_filename.name.replace("_cleaned.jsonl", ".pdf") )
# pdf_doc = converter.convert(source=pdf_filepath).document

# ## TODO load meta_data from Haystack, remove workaround here with  minimum version of metadata
# tt = DoclingParser(doc=pdf_doc, meta_data={"source": pdf_filename},
#     # start_page=0, end_page=3, #3831, end_page=3832,
#       double_notes=False      # enable footnotes
#     ).run()
# tt
# chunk_iter = chunker.chunk(dl_doc=tt[0])

In [ ]:
chunker.chunk?

#### org (imporve loader + chunking)

In [ ]:

# print( "Number of documents to process:", len(os.listdir(DOCS_DIR)) )
print("Number of documents to process:", len(search_path) )
start_time = time.time()

EXPORT_TYPE = ExportType.DOC_CHUNKS

# load tokenizer
embed_model =  "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(embed_model),
    max_tokens=256, # max tokens for MiniLM-l6-v2, set here explicitly
    # standardize input sizes of chunks for Llama models
    padding=True, # add zero as extra tokens to too short sequences so that they have the same length as other chunks
    truncation=True, # truncates too long sequences (> max_tokens). If False, they will be split into multiple chunks
)

## init chunker - based on hierachical chunker but also considers max token leng, merge smaller chunks, except when at end of paragraph (merge_peers=True)
chunker = HybridChunker(
    tokenizer=tokenizer,
        # max_tokens=256, # max tokens for MiniLM-l6-v2, set here explicitly
        # chunk_overlap=0, # no overlap between chunks, as we use merge_peers to merge smaller chunks and avoid splits in sentences
    split_by_sentence=True, # split by sentence first before merging smaller chunks, to avoid splits in sentence middle
    merge_peers=True,  # optional, defaults to True
)


# convert the different layouts of the pdf files into unified markdown format incl. sub/section titles, tables, caption text etc
# for pdf_filename in os.listdir(DOCS_DIR):
for pdf_filename in search_path:
    # if pdf_filename.endswith(".pdf"):

        md_filename = f"{Path(pdf_filename).stem}.md"

        #NOTE dummy for dev and optimizing chunking and text7doc cleaning
        # pdf_filepath = os.path.join(DOCS_DIR, Path(pdf_filename))
        pdf_filepath = os.path.join(DOCS_DIR, pdf_filename.name.replace("_cleaned.jsonl", ".pdf") )


        md_filepath = os.path.join(PARSED_TEXT_DIR, Path(md_filename))
        cleaned_jsonl_filepath = md_filepath.replace(".md", "_cleaned.jsonl")

        # if os.path.exists(cleaned_jsonl_filepath):
        #     print(f"Cleaned jsonl file already exists: '{cleaned_jsonl_filepath}'")
        #     continue

        print(f"\nFetching: {pdf_filename}")

             
        pdf_doc = converter.convert(source=pdf_filepath).document

        # document-wise translation and cleaning

        # TODO translation impl in docling parser

        # ## TODO load meta_data from Haystack, remove workaround here with minimum version of metadata
        # pdf_doc = DoclingParser(
        #     doc=pdf_doc, 
        #     meta_data={"source": pdf_filename},
        #     # double_notes=False      # enable footnotes
        # ).run()

        # loader = DoclingLoader(
        #     pdf_filepath,
        #     export_type=EXPORT_TYPE,
        #     chunker=HybridChunker(tokenizer=embed_model),
        # )



        # chunk-wise cleaning and contextualization

        # chunking
        chunk_iter = chunker.chunk(dl_doc=pdf_doc)
        chunks = list(chunk_iter)

        for i, chunk in enumerate(chunks):
            
            print(f"=== {i} ===")
            text_tokens = tokenizer.count_tokens(chunk.text)
            print(f"chunk.text ({text_tokens} tokens):\n{chunk.text!r}")

            # text cleaning
            text_cleaned = chunk.text

            print("NOTE: using linebreak handling already before saving as MD as it improves chunking with DoclingLoader (eg. partly avoids chunk splits within sentences)")
            ## ISSUE: pdf_text_no_refs has linebreak symbols "\n" which are used as potential chunk breaks by DoclingLoader
            # eg. Koks et al 2022: \n occurd in pdf_text_no_refs_urls will be used by DoclingLoader as potenital chunk breaks --> issue: sometimes splits in sentence middle
            text_cleaned = re.sub(r"([^\s-])\n([^\s-])", r"\1 \2", text_cleaned) # replace linebreak symbols when they occur just once, with whitespace (two linebreaks - probably new subsection)
            
            # TODO check if ital/esp text is already getting better with bruise code
            # text_cleaned = text_cleaned.replace("/\n{2,}/g", "\n")  # replace multiple linebreaks to just one (e.g. before subsection)
            
            # Matches \n not preceded or followed by \n
            text_cleaned = re.sub(r"\s+", " ", text_cleaned)  # replace >1 whitespaces with single whitespace
            text_cleaned = re.sub(r"([^\s-])- ([^\s-])", r"\1-\2", text_cleaned)  # remove hyphens in the middle of lines
    
            ## Translation
            src_language_doc = langdetect.detect(pdf_filename.stem.replace(".pdf", "").lower())  # lower case improves language detection

            if src_language_doc != "en":
                supported_languages = ["fr", "de", "es", "it", "itc", "nl"]
                if src_language_doc not in supported_languages:
                    print(f"Unsupported source language: {src_language_doc}. Continue with extraction on original text")
                    continue 

                print(f"Translating {src_language_doc} --> en")
                # for i, chunk in enumerate(text_cleaned):
                #     text_cleaned[i].page_content = tm.translate_2_english(src_language_doc, chunk.page_content)
                text_cleaned = tm.translate_2_english(src_language_doc, text_cleaned)        

            chunk.text = text_cleaned

            # apply contextualization
            ser_text = chunker.contextualize(chunk=chunk)
            ser_tokens = tokenizer.count_tokens(ser_text)
            print(f"chunker.contextualize(chunk) ({ser_tokens} tokens):\n{ser_text!r}")
            print()
            ###############   

        # print(f"Saving parsed and cleaned document as jsonl to: {cleaned_jsonl_filepath}")
        # # md_text_cleaned.save_as_markdown(cleaned_md_filepath)
        # # dc.save_doc_to_jsonl(text_cleaned, cleaned_jsonl_filepath)
        # with open(cleaned_jsonl_filepath, "w") as f: #, encoding="utf-8") as f:
        #         for d in text_cleaned:
        #             print(d)
        #             f.write(d.json() + '\n')



        # # FIXME remove workaround of saving pdf as markdown and reading it again as Docling.Document
        # with open(md_filepath, "w", encoding="utf-8") as f:
        #     f.write(pdf_text_no_refs_urls)

        
        # print("Converting Markdown to text...")
        # # # FIXME with DocLoader
        # loader = DoclingLoader(
        #     md_filepath,
        #     export_type=EXPORT_TYPE,
        #     chunker=HybridChunker(tokenizer=embed_model),
        # )
        # md_doc = loader.load()
        # # md_text = converter.convert(md_filepath)  #org, ISSUE: converter splits text not at dots --> create incomplete texts (e.g.. losses the end) and writes it to cleand.md where it seems to be contiuing with new sentence but without "dot"


        # for i, c in enumerate(md_doc_splits):
        #     # TODO doc .cleaning needs improvement
        #     c = c.page_content
            
        #     #c = c.replace(r"\n", r" ")   # Isssue replaces also multipelinebreaks eg before subsection
        #     c = re.sub(r"([^\s-])\n([^\s-])", r"\1 \2", c) # replace linebreak symbols when they occur just once, with whitespace (two linebreaks - probably new subsection)
        #     c = c.replace("/\n{2,}/g", "\n")  # remove linebreaks only when they occurred just once, but not for multiple linebreaks (e.g. before subsection)
        #     # Matches \n not preceded or followed by \n
        #     # c = re.sub(r"(?<!\n)\n(?!\n)", r"\n", c)  # remove linebreaks only when the yoccured just once, but not for multiple linebreaks (e.g. before subsection)
        #     c = re.sub(r"\s+", " ", c)  # replace >1 whitespaces with single whitespace
        #     # c = c.replace(r"\w*- ", "\w*-", c)  # removes any word followed by "-"
        #     # c = re.sub(r"([^\s-])-\n([^\s-])", r"\1\2", c)  # remove hypen and linebreaks TODO test with koks sentences
        #     c = re.sub(r"([^\s-])- ([^\s-])", r"\1-\2", c)  # remove hypens in the middle of lines


        #     # print("--> NEW:" ,c)
        #     # writing cleaned text back to doc
        #     md_doc_splits[i].page_content = c

        # text_cleaned = md_doc_splits
        # print(text_cleaned)




end_time = time.time() - start_time
print(f"Parsing and cleaning done. Time elapsed: {end_time:.2f} seconds.")


# visual check of removed items
# TODO make as document_cleaning function: print removed items with largest number of chars first
# ## NOTE. high number of chars == more potentially actual text body

# text_items_removed = sorted(text_items_to_drop_visualization, key=lambda x: -x[0])
# for i in text_items_removed[:50]:
#     print(i) # -->  also subsection titles were removed partly


# %%
question_1 = "Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location, and the type of damage."
question_2 = "Is the location of each affected or damaged critical infrastructure correctly identified?"

# %%




gc.collect()
torch.cuda.empty_cache() 
torch.no_grad()
print(torch.cuda.memory_reserved() / 1e9)




In [ ]:

# textitem   # chunks 0 qnd 14


# ref
# listitem
## parent=RefItem(cref='#/groups/9'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.LIST_ITEM: 'list_item'>, prov=[ProvenanceItem(page_no=8,

In [ ]:

# # EXPORT_TYPE = ExportType.DOC_CHUNKS
# # loader = DoclingLoader(
# #     md_filepath,
# #     export_type=EXPORT_TYPE,
# #     chunker=HybridChunker(tokenizer=embed_model),
# # )
# # md_doc = loader.load()
# # if EXPORT_TYPE == ExportType.DOC_CHUNKS:
# #     md_doc_splits = md_doc
# # else:
# #     raise ValueError(f"Unexpected export type: {EXPORT_TYPE}")
# ### --> splits md text still at sentecne center, not at dots, sometimes.





# from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
# from transformers import AutoTokenizer
# from docling.document_converter import DocumentConverter

# # call doc
# # doc = DocumentConverter().convert(source=md_filepath).document
# doc =  DocumentConverter(
#     allowed_formats=[InputFormat.PDF, InputFormat.MD],
#     # format_options={
#     #     InputFormat.PDF: FormatOption(
#     #         pipeline_cls=StandardPdfPipeline,
#     #         pipeline_options=pipeline_options,
#     #         backend=PyPdfiumDocumentBackend,
#     #     ),
#     # },
# ).convert(source=md_filepath).document

# # load tokenizer
# tokenizer = HuggingFaceTokenizer(
#     tokenizer=AutoTokenizer.from_pretrained(embed_model),
#     max_tokens=256, # 256 max tokens per chunk, # optional, by default derived from `tokenizer` for HF case
#     # standardize input size for Llama models, 
#     padding=True, # add zero as extra tokens to too short sequences so that they have the same length as other chunks
#     truncation=True, # truncates too long sequences (> max_tokens). If False, they will be split into multiple chunks
# )

# ## init chunker - based on hierachical chunker but also considers max token leng, merge smaller chunks, except when at end of paragraph (merge_peers=True)
# chunker = HybridChunker(
#     tokenizer=tokenizer,
#     merge_peers=True,  # optional, defaults to True
    
# )
# chunk_iter = chunker.chunk(dl_doc=doc)
# chunks = list(chunk_iter)

# for i, chunk in enumerate(chunks):
#     print(f"=== {i} ===")
#     txt_tokens = tokenizer.count_tokens(chunk.text)
#     print(f"chunk.text ({txt_tokens} tokens):\n{chunk.text!r}")

#     ser_txt = chunker.contextualize(chunk=chunk)
#     ser_tokens = tokenizer.count_tokens(ser_txt)
#     print(f"chunker.contextualize(chunk) ({ser_tokens} tokens):\n{ser_txt!r}")

#     print()

# # from langchain_experimental.text_splitter import SemanticChunker, RecursiveCharacterTextSplitter
# # from langchain_openai.embeddings import OpenAIEmbeddings 
# # # Split the document into chunks
# # text_splitter = SemanticChunker(OpenAIEmbeddings()) 
# # # text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
# # chunks = text_splitter.create_documents([md_doc.page_content])
# # # Print the first chunk
# # print(chunks[0].page_content)
# ###################        
# # md_doc = loader.load()
# # md_doc_splits[7].page_content

'The German states most af- fected include Rhineland-Palatinate (Rheinland-Pfalz), with damage to the Ahr River valley (Ahrtal), several regions in the Eiffel National Park, to the city of Trier. Flooding in Belgium was concentrated in the Vesdre River valley (dis- tricts of Pepinster, Ensival and Verviers), the Meuse River valley (Maaseik, Liége), the Gete River valley (Herk-de-Stad and Halen) and southeast Brussels (Wavre). The Netherlands experienced ﬂooding, mostly concentrated in the southern district of Limburg. In total, at least 220 casualties have been reported, with insured loss estimates of approximately EUR 150 million–EUR 250 million in the Netherlands (Ver- bond voor Verzekeraars, 2022), ∼ EUR 2.2 billion in Bel- gium (Assuralia, 2022) and ∼ EUR 8.2 billion (GDV, 2022) in Germany. The event caused major damages to residential and commercial structures and to many critical infrastruc- ture (CI) assets.'


### bump from transformers to Langchain + Memory passing


In [ ]:

# class Memory:
#     # Source. https://medium.com/@jagadeesan.ganesh/mastering-llm-ai-agents-building-and-using-ai-agents-in-python-with-real-world-use-cases-c578eb640e35
#     def __init__(self):
#         self.memory_store = []

#     def remember(self, interaction):
#         self.memory_store.append(interaction)

#     def recall(self):
#         return " ".join(self.memory_store)
# from langchain_core.output_parsers import StrOutputParser
# from langchain_core.prompts import PromptTemplate

# model_name = "meta-llama/Llama-3.1-8B-Instruct"


# # prompt_template = "Tell me a {adjective} joke"
# # prompt = PromptTemplate(input_variables=["adjective"], template=prompt_template)
# model = AutoModelForCausalLM.from_pretrained(
#                 model_name,
#                 dtype="auto", # None ,# test for CU12.6, torch.29.1 #"auto",
#                 # max_memory={0: "2GB", 1: "10GB"},  # distribute memory across GPUs
#             )
# # chain = prompt | model #| StrOutputParser()

# # chain.invoke("your adjective here")


# # Define a simple prompt for the agent
# template = """
# You are an AI assistant with expertise in data analysis and automation. Answer the following question:
# Question: {question}
# """

# # Set up the prompt and LLM chain
# prompt = PromptTemplate(template=template, input_variables=["question"])
# chain = prompt | model | StrOutputParser() # LLMChain(prompt=prompt, llm=llm)

# # # # Example query
# # query = "What is the impact of AI in healthcare?"
# # response = chain.run(question=query)
# # print(f"Agent Response: {response}")


# from langchain.agents import create_agent

# agent = create_agent(
#     model=model_name,
#     model_provider="huggingface",
#     # tools=[get_weather],
#     system_prompt="You are a helpful assistant",
#     temperature=0.0,
#     max_tokens=1024,
# )

# result = agent.invoke(
#     {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
# )
# print(result["messages"][-1].content_blocks)


##  CI impact extraction


In [ ]:
# %%
# Settings
model_name = "meta-llama/Llama-3.1-8B-Instruct"

time0 = time.time()

print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# print(os.environ["CUDA_VISIBLE_DEVICES"])

# clean up before applying CUDA
gc.collect()
torch.cuda.empty_cache() 
print(torch.cuda.memory_reserved() / 1e9)
torch.no_grad()

## init LLM extraction models
decoder_model_1 = em.DecoderModelCaching(
    model_name,
    em.load_prompt_template(template_filename="short_static_llama3_NER.txt",)
)

decoder_model_2 = em.DecoderModelCaching(
    model_name,
    em.load_prompt_template(template_filename="short_static_llama3_NER_geollm_step2.txt",)
)


gc.collect()
torch.cuda.empty_cache() 
torch.no_grad()
print(torch.cuda.memory_reserved() / 1e9)

# print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# print(os.environ["CUDA_VISIBLE_DEVICES"])

# CI-GEO pairs
geolocs_cache = geonamescache.GeonamesCache()
countries = geolocs_cache.get_countries()
ci_geo_countries = [*u.gen_dict_extract(countries, 'name')]



## init outputs
df_resp_step1_all = pd.DataFrame()
df_resp_step2_all = pd.DataFrame()
responses_error_list = []
df_ci_cases_not_grouped = pd.DataFrame()
df_geollama_response = pd.DataFrame()


if test_mode:
    search_path = docs_list_sample
    print("Test mode is ON. Using only a small sample of documents for testing.")
else:
    search_path = glob(str(Path(PARSED_TEXT_DIR, "*cleaned.jsonl")))




## Start CI impact extraction
for file_no, filename in enumerate(search_path):

    time1 = time.time()

    src_language_nonengl = None 

    no_documents = len(search_path)
    filepath = Path(filename)
    filename_stem = filepath.stem


    ## load doc (from jsonl)
    try:
        doc = dc.load_doc_from_jsonl(filepath)
    except FileNotFoundError:
        print(f"File not found: {filepath}. Skipping this document.")
        continue



    print(f"\n\n ######## -------- Processing document [{file_no+1}/{no_documents}]: {filepath.name} -------- ######## \n")

    ## extract authors, publication year and title 
    author, year, title = dc.extract_citation_info(filename_stem)
    citation = f"{author} {year}".replace("  ", " ").strip()
    title = title.replace(" - ", "").replace("_cleaned", "").strip()



    # init dfs to store interim results for each doc
    df_resp_step1 = pd.DataFrame(
        columns=[
            "citation_id",
            "chunk_id",
            "infrastructure_type",
            "damage",
            "damage_value",
            "location",
            "ci_entity",
            "geo_entity",
            "chunk_text"
        ]
    )
    df_resp_step2 = pd.DataFrame(
        columns=[
            "citation_id",
            "chunk_id",
            "infrastructure_type",
            "infrastructure_group",
            "damage",
            "damage_value",
            "location",
            "ci_entity",
            "geo_entity",
            "coord_potential_locations",
            "chunk_text"
        ]
    )
    
    for chunk_no, chunk in enumerate(doc):

        # init dfs for interim results for each chunk
        ## TODO make as pydantic class with fixed attributes
        df_ci_geo_chunk = pd.DataFrame(
            columns=[
                "citation_id",
                "ci_entity",
                "geo_entity",
                "chunk_text",
                "token_distance",
            ]
        )
    
        df_geollama_response = pd.DataFrame()

    
        print(f"\n\nProcessing chunk no. {chunk_no+1} / {len(doc)} of document: {filepath.name}")

        print("Chunk text: ", chunk.page_content)
     
        ## preprocess  TODO move to document cleaning workflow + dc.funcs
        # NOTE currently done in doc. cleaning (when writign to _cleaned.jsonl)
        # chunk.page_content = chunk.page_content.replace("\n", " ")
        # chunk.page_content = chunk.page_content.replace("- ", "-") # TODO test if ("- ", "") is better


        print("\nGetting geolocations of CI assets ")       
        ## get most likely geolocation for each CI entity based on distance between tokens
        nlp_chunk = nlp(chunk.page_content)
        all_ents = [ent for ent in nlp_chunk.ents]
        ci_type_ents = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE", "FAC"]]


        # check if chunk contains CI_TYPE entities
        if len(ci_type_ents) > 0:

            # iterate over all entities within chunk
            for ent_idx in range(len(all_ents)):
                # when entity is CI_TYPE or FAC (i.e. buidling, airports, highways) do following ...
                if all_ents[ent_idx].label_ in ["CI_TYPE", "FAC"]:
                    ci_idx = ent_idx

                    ## .. calculate distances between CI_TYPE entity and  all GEO entities in chunk based on index position
                    distance_list = []
                    idx_in_chunk = []
                    try:
                        for ent_idx in range(len(all_ents)):

                            # TODO calc distances between CI_TYPE ~ GEO entities based on word numbers and not entities (ie tokens)
                            if all_ents[ent_idx].label_ in ["GPE", "LOC"]:

                                ## check that GPE,LOC are not countries (too coarse info CI-GEO pair)
                                ## of GPE/LOC is country -> proceed with next GPE/LOC 
                                if all_ents[ent_idx].text in ci_geo_countries:
                                    continue

                                geo_idx = ent_idx
                                dist_ent_pair = np.abs(ci_idx - geo_idx)
                                distance_list.append(dist_ent_pair)
                                idx_in_chunk.append((ent_idx))
                                closest_pair_idx = np.argmin(distance_list)  # idx of closest GEO entity
                                distance_closest_pair = distance_list[closest_pair_idx]

                        threshold = 5  # max token distance between CI_TYPE and GEO entity
                        if distance_closest_pair > threshold:
                            # print(
                            #     f""" Token distance is too large between CI_TYPE/FAR and next GEO entity which is of {distance_closest_pair} [token distance] > {threshold} [max. token distance] """
                            # )
                            continue
                        else:
                            pass

                        ## write as dict entry incl chunk_id, ci_entity, geo_entity, distance
                        result_dict = {
                            "citation_id": citation,
                            "chunk_text": chunk.page_content,
                            "ci_entity": all_ents[ci_idx].text,
                            "ci_entity_label": all_ents[ci_idx].label_,
                            "geo_entity": all_ents[idx_in_chunk[closest_pair_idx]].text,
                            "geo_entity_label": all_ents[idx_in_chunk[closest_pair_idx]].label_,
                            "token_distance": distance_closest_pair,
                        }
                        df_ci_geo_chunk = pd.concat(
                            [  df_ci_geo_chunk, pd.DataFrame([result_dict])], ignore_index=True
                        )

                    except (IndexError, NameError) as e:
                        continue
        else:
            print("No CI_TYPE entities found in this chunk. Going to next chunk")
            continue

        
        ## post-process of DF CI-GEO pairs for each chunk
        unique_ci_geo_pairs = df_ci_geo_chunk.drop_duplicates(
            subset=["citation_id", "ci_entity", "geo_entity","chunk_text"])
        print("number of duplicates to remove:", len(df_ci_geo_chunk) - len(unique_ci_geo_pairs))

        df_ci_geo_chunk = df_ci_geo_chunk.drop_duplicates(
            subset=["citation_id", "ci_entity", "geo_entity", "chunk_text"]
            )# .reset_index(drop=True, inplace=True)



        print(f"\n Text-2-Data:")

        ## apply decoder on each chunk in document
        ## TODO replace iteration by loading entire document and use recursive chunking from langchain
        time2 = time.time()


        # clean up before applying CUDA
        gc.collect()
        torch.cuda.empty_cache() 
        torch.no_grad()
        # print(torch.cuda.memory_reserved() / 1e9)
    
        print(f"Starting geoparsing")

        ## Start geoparsing with geollama to verify Location        
        
        # extract locations
        # for d in tqdm(chunk.page_content):
        resp = geo_llama.geoparse(chunk.page_content)
        # TODO add here geollama prompt as var
        # TODO check if useful in model.py to set : model.use_checkpointing = True or  model.gradient_checkpointing_enable()
        
        # save results
        df_geollama_resp = pd.DataFrame(resp[0:])
        df_geollama_resp["citation_id"] = citation
        df_geollama_resp["chunk_id"] = chunk_no
        df_geollama_resp["chunk_text"] = chunk.page_content
        
        # maybe empty response
        try:
            print("Removing duplicated locations and countries from geollama response to keep only more specific location info")

            df_geollama_resp["name"] = list(set(df_geollama_resp["name"]))  # rm dublicates
            df_geollama_resp = df_geollama_resp[~df_geollama_resp["name"].isin(ci_geo_countries)]  # rm countries
            print("Toponyms from geollama\n", df_geollama_resp["name"])
        except:
            pass

        # saving all geollama repsonses , even when they cannot processed furthernfor later analysis
        df_geollama_response = pd.concat([df_geollama_response, df_geollama_resp], ignore_index=True)
        

        try:
            locations_per_chunk = df_geollama_resp["name"].to_list()
            lats_per_chunk = df_geollama_resp["latitude"].to_list()
            lons_per_chunk = df_geollama_resp["longitude"].to_list()
            RAGestimated_per_chunk = df_geollama_resp["RAG_estimated"].to_list()
        except:
            print("No locations extracted by geollama for this chunk. Going to next chunk\n", resp[0:])
            continue
        
        # sanity check that only cases with location infos are considered for further processing      
        if len(locations_per_chunk) == 0:
            print("geollama did not find any potential locations found for this chunk. Continue with next chunk")
            continue

        ## Input for LLM 1  -1st round
        if df_ci_geo_chunk.empty:
            context = [
                {
                    "text": chunk.page_content,
                    # "citation": citation,
                    # "title": filename_stem,
                    "entity_recognition": None,
                },
            ]

        else:  # TODO dissolve if else clause by making it in ci_locations: if df_ci_geo.chunk=j, xx, else None
            context = [
                {
                    "text": chunk.page_content,
                    # "citation": citation,
                    # "title": filename_stem,
                    "entity_recognition": df_ci_geo_chunk,
                },
            ]


        # print(os.environ["CUDA_VISIBLE_DEVICES"])
        # os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
        # print(os.environ["CUDA_VISIBLE_DEVICES"])

        # load LLM_1 prompt template
        static_prompt = em.load_prompt_template(template_filename="short_static_llama3_NER.txt")
        dynamic_prompt = em.load_prompt_template(template_filename="short_dynamic_llama3_NER.txt")
        # template_1 = em.load_prompt_template(template_filename="ci_loc_direct_impacts_geollama_step_1.txt")

        # apply LLM 1
        response = decoder_model_1.generate_response(
            question=question_1, context=context, # chunk_id=j
            static_prompt = static_prompt,
            dynamic_prompt=dynamic_prompt,
            max_new_tokens = 2048
        )
        # print(type(response ))
        # print(response)
        
        ## postprocess response
        try:
            try:
                df_resp = pp.postprocess_response(response[0])
            except Exception as e:
            # except (IndexError, ValueError) as e:
                df_resp = pp.postprocess_response(response)
            
            # save LLM response for each chunk  in dataframe for each document
            df_resp["citation_id"] = citation 
            df_resp["chunk_id"] = chunk_no  # add chunk id as identifier
            df_resp["ci_entity"] = df_ci_geo_chunk["ci_entity"] 
            df_resp["geo_entity"] = df_ci_geo_chunk["geo_entity"]
            df_resp["coord_potential_locations"] = str(dict(zip(locations_per_chunk, zip(lats_per_chunk, lons_per_chunk, RAGestimated_per_chunk)))) # INTERIM for verification of lat, lon 
            df_resp["chunk_text"] =  context[0]["text"]  # add (translated) chunk text for tracing back LLM response

            if not len(df_resp):
                print("LLM response is empty. Continue with next chunk.\n Response was:", response)
                continue

            # store reponse for chunk to interim df
            df_resp_step1 = pd.concat([df_resp_step1, df_resp], ignore_index=True)


        except (IndexError, ValueError) as e:
            print(f"Cannot add response: {e},\nFaulty response (before postprocessing):", response)
            # faulty response: e.g.  .., "location": "V" on satellite and online on radar"}, { ...}, {}
            responses_error_list.append({
                "citation_id": citation,
                "chunk_id": chunk_no,
                "response": response,
                "error": str(e)
            })
            print("Continue with next chunk\n")
            continue


        ## group Ci types into subgroups, 
        # TODO make nicer when df is empty
        if df_resp.isna().sum().sum() == 0:
            print("No infrastructure types extracted for this chunk, skip grouping into subgroups. Go to next chunk")
            continue

        df_resp = pp.group_ci_types(df_resp, "infrastructure_type", "infrastructure_group", ci_patterns)
        ## store cases which could not be grouped
        df_ci_cases_not_grouped = pd.concat([df_ci_cases_not_grouped, df_resp[df_resp['infrastructure_group'].isna()]], ignore_index=True)
        ## keep only records which are actually about CI (e.g., not theatre, stadion ..)
        df_resp.dropna(subset=["infrastructure_group"], inplace=True)
    

        # clean up after each chunk
        gc.collect()
        torch.cuda.empty_cache()  # mainly needed after training, small effect when LLM applied only for inference
        torch.no_grad()

        print(f"\n   Processing time for chunk {chunk_no+1} STEP 1: {np.round((time.time() - time2) / 60, 1)} minutes\n")

        print("STEP 2")
        print("KEEPING location info from step 1 for further improvement")
        df_locs_org = df_resp.copy() 


        # clean up before applying CUDA
        gc.collect()
        torch.cuda.empty_cache() 
        print(torch.cuda.memory_reserved() / 1e9)
        torch.no_grad()

        # print(os.environ["CUDA_VISIBLE_DEVICES"])
        # os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
        # print(os.environ["CUDA_VISIBLE_DEVICES"])


        ## Input for LLM - 2nd round (geollama)
    
        # print(f"\nCHECKING input.text, df_resp, geollama resp \n{chunk.page_content}\n{df_resp},\n{locations_per_chunk}")
        static_prompt_geollama = em.load_prompt_template(template_filename="short_static_llama3_NER_geollm_step2.txt")
        dynamic_prompt_geollama = em.load_prompt_template(template_filename="short_dynamic_llama3_NER_geollm_step2.txt")
        
        context = [
            {
                "text": chunk.page_content,
                "citation": citation,
                "title": filename_stem, 
                "previous_response": df_resp,
                "potential_locations": locations_per_chunk,  # list of 1 or multiple location strings (no countries)      
                # "coordinates_potential_locations": list(zip(locations_per_chunk, lats_per_chunk, lons_per_chunk, RAGestimated_per_chunk)),
            },
        ]
        
        ## LLM 1 - step2
        response = decoder_model_2.generate_response(
            question=question_2, context=context, # chunk_id=j
            static_prompt = static_prompt_geollama,
            dynamic_prompt = dynamic_prompt_geollama,
            max_new_tokens = 2048
        )

        ## postprocess response
        try:
            try:
                df_resp_2 = pp.postprocess_response(response[0])
            except Exception as e:
            # except (IndexError, ValueError) as e:
                df_resp_2 = pp.postprocess_response(response)

            # save LLM response for each chunk  in dataframe for each document
            df_resp_2["citation_id"] = citation
            df_resp_2["chunk_id"] = chunk_no  # add chunk id as identifier
            df_resp_2["ci_entity"] = df_ci_geo_chunk["ci_entity"] 
            df_resp_2["geo_entity"] = df_ci_geo_chunk["geo_entity"]
            df_resp_2["coord_potential_locations"] = str(dict(zip(locations_per_chunk, zip(lats_per_chunk, lons_per_chunk, RAGestimated_per_chunk)))) # INTERIM for verification of lat, lon 
            df_resp_2["infrastructure_type_org"] = df_locs_org["infrastructure_type"] 
            df_resp_2["damage_org"] = df_locs_org["damage"] 
            df_resp_2["damage_value_org"] = df_locs_org["damage_value"] 
            df_resp_2["locations_org"] = df_locs_org["location"] 
            df_resp_2["chunk_text"] =  context[0]["text"]  # add (translated) chunk text for tracing back LLM response

            # collect resps for each doc
            print("CREATED final LLM response (STEP 1 & 2) successfully")
            df_resp_step2 = pd.concat([df_resp_step2, df_resp_2], ignore_index=True)
            # print(df_resp_2)

        except (IndexError, ValueError) as e:
            try: 
                print(f"Cannot add response: {e},\nFaulty response (before postprocessing):", response)
                # faulty response: e.g.  .., "location": "V" on satellite and online on radar"}, { ...}, {}
                responses_error_list.append({
                    "citation_id": citation,
                    "chunk_id": chunk_no,
                    "response": response, #.replace('\n', ''),
                    "error": str(e)
                })
            except:
                pass

        print(f"\n   Processing time for chunk {chunk_no} STEPs 1 & 2: {np.round((time.time() - time2) / 60, 1)} minutes\n")

        # clean up before applying CUDA
        gc.collect()
        torch.cuda.empty_cache() 
        print(torch.cuda.memory_reserved() / 1e9)
        torch.no_grad()

        print(os.environ["CUDA_VISIBLE_DEVICES"])



    print(f"\n   Processing time for document {citation} STEP 1&2: {np.round((time.time() - time1) / 60, 1)} minutes\n")

    print(f"Safety: saving responses (Step 1 + 2) for doc: {citation} ")
    df_resp_step1.to_csv(f"llm1_geollm_step1_{citation}.csv", encoding='utf-8', index=False)
    df_resp_step2.to_csv(f"llm1_geollm_step2_{citation}.csv", encoding='utf-8', index=False)

    # clean up after each document
    gc.collect()
    torch.cuda.empty_cache()  # mainly needed after training, small effect when LLM applied only for inference
    torch.no_grad()

    # collecting all docs
    df_resp_step1_all = pd.concat([df_resp_step1_all, df_resp_step1], ignore_index=True)
    df_resp_step2_all = pd.concat([df_resp_step2_all, df_resp_step2], ignore_index=True)



print(f"\n\n ######## -------- CI impact extraction took {(time.time() - time0) / 60} minutes -------- ######## \n\n")


# %%


# %% [markdown]
# ### Finish run

# %%
print("Where CI types could not be grouped:\n", df_ci_cases_not_grouped)


# %%

df_resp_step2.info()

# %%
print(len(df_resp_step1))
unique_ci_geo_pairs = df_resp_step1.drop_duplicates()
print("number of duplicates to remove:", len(df_resp_step1) - len(unique_ci_geo_pairs))

df_resp_step1_nodupl = df_resp_step1.drop_duplicates( )# .reset_index(drop=True, inplace=True)
print(len(df_resp_step1_nodupl))
df_resp_step1_nodupl

# %%
print(len(df_resp_step2))
unique_ci_geo_pairs = df_resp_step2.drop_duplicates()
print("number of duplicates to remove:", len(df_resp_step2) - len(unique_ci_geo_pairs))

df_resp_step2_nodupl = df_resp_step2.drop_duplicates( )# .reset_index(drop=True, inplace=True)
print(len(df_resp_step2_nodupl))
df_resp_step2_nodupl


print(df_resp_step2.infrastructure_group.isna().sum())  # mostly cases which are not CI (theater, stadion..)
print(df_resp_step2.infrastructure_group.value_counts()) # four most common subgroups seems to be correct
# df_pred.infrastructure_group.unique()



gc.collect()
torch.cuda.empty_cache() 
torch.no_grad()
print(torch.cuda.memory_reserved() / 1e9)
# %%
print(torch.cuda.memory_reserved() / 1e9)


# %%
print("Chunk with erroneous responses:", responses_error_list.__len__())
df_responses_error = pd.DataFrame(responses_error_list)
# df_responses_error#.tail(3)


# %%
# df_responses_all_step2#.tail(3)

# %% [markdown]
# ### Saving

# %%
# PATH_LLM_DATA: Path = Path(s.PATH_DATA /"llm_outputs/")
# LLM_DATA_FILENAME: str = "llm_1_updprompt_distanceNER.csv"

# OUTPUT_LLM1_FILEPATH =  Path(PATH_LLM_DATA / LLM_DATA_FILENAME)#.replace(".csv", "_v2.csv"))
# OUTPUT_LLM1_FILEPATH 

# %%
safety_df = df_resp_step2_all.copy()

# save LLM 1 output to disk along with prompt text
if not os.path.isfile(OUTPUT_LLM1_FILEPATH):

    print(f"Saving prompt, LLM response and erroneous responses [.txt, .csv] to {OUTPUT_LLM1_FILEPATH} ...")

    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_staticgeol_{OUTPUT_LLM1_FILEPATH.stem}.txt", "w") as f:
        f.write(static_prompt.render(context=context, question=question_1))
    try:
        with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_dynamicgeol_{OUTPUT_LLM1_FILEPATH.stem}.txt", "w") as f:
            f.write(dynamic_prompt.render(context=context, question=question_1))
    except Exception as e:
        print("UndefinedError: dynamic_prompt has probably no df_ci_geo (it is empty)")   
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_staticgeol_{OUTPUT_LLM1_FILEPATH.stem}.txt", "w") as f:
        f.write(static_prompt_geollama.render(context=context, question=question_2))
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_dynamicgeol_{OUTPUT_LLM1_FILEPATH.stem}.txt", "w") as f:
        f.write(dynamic_prompt_geollama.render(context=context, question=question_2))

    df_resp_step1_all.to_csv(f"{OUTPUT_LLM1_FILEPATH.stem}_step1.csv", encoding='utf-8', index=False)
    df_resp_step2_all.to_csv(f"{OUTPUT_LLM1_FILEPATH.stem}_step2.csv", encoding='utf-8', index=False)
    df_responses_error.to_csv(OUTPUT_LLM1_FILEPATH.parent / f"errors_{OUTPUT_LLM1_FILEPATH.stem}.csv", encoding='utf-8', index=False)

elif os.path.isfile(OUTPUT_LLM1_FILEPATH) and not os.path.isfile(OUTPUT_LLM1_FILEPATH.parent / f"{OUTPUT_LLM1_FILEPATH.stem}_v2.csv"):

    print(f"Output file {Path(OUTPUT_LLM1_FILEPATH).stem} already exists. Saving as {OUTPUT_LLM1_FILEPATH.stem}_v2 to avoid overwriting ...")

    # If the original files exists but the v2 file doesn't, create the v2 file
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_staticgeol_{OUTPUT_LLM1_FILEPATH.stem}_v2.txt", "w") as f:
        f.write(static_prompt.render(context=context, question=question_1))
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_dynamicgeol_{OUTPUT_LLM1_FILEPATH.stem}_v2.txt", "w") as f:
        f.write(dynamic_prompt.render(context=context, question=question_1))
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_staticgeol_{OUTPUT_LLM1_FILEPATH.stem}_v2.txt", "w") as f:
        f.write(static_prompt_geollama.render(context=context, question=question_2))
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_dynamicgeol_{OUTPUT_LLM1_FILEPATH.stem}_v2.txt", "w") as f:
        f.write(dynamic_prompt_geollama.render(context=context, question=question_2))
    
    df_resp_step1_all.to_csv(Path(OUTPUT_LLM1_FILEPATH.parent, f"{OUTPUT_LLM1_FILEPATH.stem}_step1_v2.csv"), encoding='utf-8', index=False)
    df_resp_step2_all.to_csv(Path(OUTPUT_LLM1_FILEPATH.parent, f"{OUTPUT_LLM1_FILEPATH.stem}_step2_v2.csv"), encoding='utf-8', index=False)
    df_responses_error.to_csv(Path(OUTPUT_LLM1_FILEPATH.parent / f"errors_{OUTPUT_LLM1_FILEPATH.stem}_v2.csv"), encoding='utf-8', index=False)

else:
    print(f"Output file {Path(OUTPUT_LLM1_FILEPATH).stem} already exists. Skip saving to avoid overwriting ...")




In [ ]:
resp.rpartition('"')[-3] + "]}"

In [ ]:
# print(f"Safety: saving responses (Step 1 + 2) for doc: {citation} ")
# df_resp_step1.to_csv(f"llm1_geollm_step1_{citation}.csv", encoding='utf-8', index=False)
# df_resp_step2.to_csv(f"llm1_geollm_step2_{citation}.csv", encoding='utf-8', index=False)

print(df_resp_step2.chunk_text[0])



## Doc. cleaning improve

In [ ]:
# c = """ 
# blublub title\n\n\n

# HERE IS NEW SUBSECTION:\n

# (large-scale) societal dis- ruptions dis - ruptions (Garschagen and Sandholz, 2018; Hallegatte et al., 2019; Fekete and Sandholz, 2021), empirical evidence on the impacts of extreme weather events on these systems is still

# Published by Copernicus Publications on behalf of the European Geosciences Union.

# E. E. Koks et al.: Flood impacts to infrastructure

# limited. This brief communication provides an overview of the observed ﬂood impacts to large-scale infrastructure sys- tems during the 2021 mid-July western European ﬂood event and how reconstruction of these large-scale systems has pro- 




# HERE IS NEW SUBSECTION

# severely damaged railway line (between the vil- lages of Spa and Pepinster) was reopened again on 3 Octo- ber 2021 (Rozendaal, 2021b). In the Netherlands, no large- scale damage has been reported to transport infrastructure. A few national highways were partly ﬂooded (e.g. the A76 in both directions) or brieﬂy closed (&lt; 3 d) because of the po- tential of ﬂooding. Most likely due to relative low-ﬂow ve- locities, damage to Dutch national road infrastructure was limited. Several railway sections were closed (e.g. the rail-

# way section between Maastricht and Liége) and some dam- age occurred to the railway infrastructure, in particular to the electronic “track circuit” devices and saturated railway em- bankments (Prorail, 2021).

# """
# #c = c.replace(r"\n", r" ")   # Isssue replaces also multipelinebreas eg before subsection
# c = re.sub(r"([^\s-])\n([^\s-])", r"\1 \2", c) # replace linebreak symbols when they occur just once, with whitespace (two linebreaks - probably new subsection)
# c = c.replace("/\n{2,}/g", "\n")  # remove linebreaks only when they occurred just once, but not for multiple linebreaks (e.g. before subsection)
# # Matches \n not preceded or followed by \n
# # c = re.sub(r"(?<!\n)\n(?!\n)", r"\n", c)  # remove linebreaks only when the yoccured just once, but not for multiple linebreaks (e.g. before subsection)
# c = re.sub(r"\s+", " ", c)  # replace >1 whitespaces with single whitespace

# # c = c.replace(r"\w*- ", "\w*-", c)  # removes any word followed by "-"
# # c = re.sub(r"([^\s-])-\n([^\s-])", r"\1\2", c)  # remove hypen and linebreaks TODO test with koks sentences
# c = re.sub(r"([^\s-])- ([^\s-])", r"\1-\2", c)  # remove hypens in the middle of lines

# c 


# # 0090- some weird breaks-
# # And some long sentences 
# # which are not separated by dots but by line breaks and hyphens 



In [ ]:
# !uv pip install "unstructured[pdf]"  # unstructured  #langchain-unstructured #langchain-community
# # # !uv add langchain
# !uv lock
# !uv sync
# from langchain_community.document_loaders import DirectoryLoader, UnstructuredFileLoader
# #from langchain.loaders import , UnstructuredFileLoader

# loader = DirectoryLoader(str(Path(DOCS_DIR)),  loader_cls=UnstructuredFileLoader, show_progress=True)
# pdf_docs = loader.load()
# # glob=glob("*.pdf"),
# print(f"Number of Documents: {len(pdf_docs)}")

# # convert the different layouts of the pdf files into unified markdown format incl. sub/section titles, tables, caption text etc
# for idx, doc in enumerate(pdf_docs, start=0):
#     print(doc)

In [ ]:
# pdf_filepath = Path("Lloyd's List 2024 - Port of Valencia reopens after devastating floods.pdf")
# # Path(DOCS_DIR, "Lloyd's List 2024 - Port of Valencia reopens after devastating floods.pdf")
# print(pdf_filepath)
# print("Remove reference section")

# # setup converter for PDF and markdown
# converter = DocumentConverter(
#     allowed_formats=[InputFormat.PDF, InputFormat.MD],
#     format_options={
#         InputFormat.PDF: FormatOption(
#             pipeline_cls=StandardPdfPipeline,
#             pipeline_options=pipeline_options,
#             backend=PyPdfiumDocumentBackend,
#         ),
#     },
# )
# pdf_text = converter.convert(pdf_filepath).document
    
# # loader = DoclingLoader("/beegfs/scratch/a-buch/_PROJECTS/data/text_sources/Lloyd's List 2024 - Port of Valencia reopens after devastating floods.pdf")  # use chunks from Docling.Loader
# # pdf_doc = loader.load()